# Detection Metric Re-analysis: Off-by-one in `align_pairs`

**Diagnostic only — this notebook does not modify `benchmark.py` or any output file.**

## Background

The current `score_heads` detection metric uses

```python
align_pairs = [(seq_len + i, i) for i in range(seq_len)]
```

On the detection sequence `primary + primary` (length `2*seq_len`), this measures, for each
position `i`, how much a head at the **second** copy of token `t_i` (query position `seq_len + i`)
attends back to the **same** token `t_i` in the first copy (key position `i`).

That is the signature of a **duplicate-token head** ("this token appeared before, here").

A canonical **induction head** (Olsson et al. 2022) does something subtly different: at the second
copy of token `A`, it attends back to the token `B` that *followed* `A` in the first copy — i.e. to
key position `i + 1`, not `i`. The corrected alignment is therefore

```python
align_pairs = [(seq_len + i, i + 1) for i in range(seq_len - 1)]
```

If the top heads change between the two metrics, the current pipeline may have been ranking a
duplicate-token head (notably **L1H11**) as the strongest "induction" head, while the true
induction heads sat lower in the ranking.

## 1. Reference: current top heads from `outputs/induction_heads/latest.json`

Before touching the model, print the detection and causal rankings already on disk, so we have a
fixed baseline to compare against.

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.benchmark import (
    build_synthetic_prompt_dataset,
    load_model,
)

latest_path = ROOT / "outputs" / "induction_heads" / "latest.json"
latest = json.loads(latest_path.read_text())
candidate_rows = latest["top_heads"]

print(f"loaded {latest_path}")
print(f"prompt_family={latest['result']['prompt_family']} "
      f"n_prompts={latest['result']['n_prompts']} "
      f"seq_len={latest['result']['seq_len']} seed={latest['result']['seed']}")

loaded /home/estevao/mechinterp/outputs/induction_heads/latest.json
prompt_family=synthetic n_prompts=6 seq_len=8 seed=0


In [2]:
ref_df = pd.DataFrame(candidate_rows)

# `top_heads` is already ranked by detection score (the topk used in the benchmark).
top5_detection = (
    ref_df.sort_values("score", ascending=False)
    .head(5)[["layer", "head", "score", "causal_selective_drop"]]
    .reset_index(drop=True)
)

top5_causal = (
    ref_df.sort_values("causal_selective_drop", ascending=False)
    .head(5)[["layer", "head", "score", "causal_selective_drop"]]
    .reset_index(drop=True)
)

print("Current top-5 DETECTION heads (ranked by attention score):")
display(top5_detection)
print("Current top-5 CAUSAL heads (ranked by causal_selective_drop):")
display(top5_causal)

Current top-5 DETECTION heads (ranked by attention score):


,layer,head,score,causal_selective_drop
0,8,5,0.912908,-0.002960
1,6,2,0.826871,0.020597
2,14,12,0.738365,0.038777
3,6,9,0.724811,0.053518
4,14,11,0.648437,0.082774


Current top-5 CAUSAL heads (ranked by causal_selective_drop):


,layer,head,score,causal_selective_drop
0,14,11,0.648437,0.082774
1,6,9,0.724811,0.053518
2,9,3,0.531135,0.048868
3,8,1,0.629790,0.044207
4,14,12,0.738365,0.038777


## 2. Load the model and rebuild the exact synthetic dataset

Same parameters as the saved run: `n_prompts=6`, `seq_len=8`, `seed=0`. CPU is fine — this is
a light forward-pass-only workload.

In [3]:
device = "cpu"
seq_len = 8
n_prompts = 6
seed = 0

model = load_model(device=device)
dataset = build_synthetic_prompt_dataset(
    model, n_prompts=n_prompts, seq_len=seq_len, seed=seed
)

print(f"n_layers={model.cfg.n_layers} n_heads={model.cfg.n_heads}")
print(f"detection_sequence length = {len(dataset[0].detection_sequence)} (= 2*seq_len)")

`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model santacoder into HookedTransformer
n_layers=24 n_heads=16
detection_sequence length = 16 (= 2*seq_len)


## 3. Inline scoring with a configurable alignment

We deliberately **do not** call `score_heads` from `benchmark.py`, because that function reads
`example.align_pairs` (the current/buggy alignment baked into each `PromptExample`). Instead we copy
the scoring logic here and pass the alignment explicitly, so we can run both metrics on the *same*
cached attention patterns and compare them fairly.

The only thing that changes between the two runs is `align_pairs`:

| metric | `align_pairs` | meaning |
|---|---|---|
| current (duplicate-token) | `[(seq_len + i, i) for i in range(seq_len)]` | 2nd copy of `t_i` &rarr; same `t_i` |
| corrected (induction) | `[(seq_len + i, i + 1) for i in range(seq_len - 1)]` | 2nd copy of `t_i` &rarr; token *after* `t_i` |

In [4]:
@torch.no_grad()
def score_heads_with_alignment(model, dataset, align_pairs):
    """Copy of benchmark.score_heads, but with an externally supplied alignment.

    Returns a [n_layers, n_heads] tensor of mean attention over the aligned (query, key) pairs
    on each example's detection_sequence.
    """
    scores = torch.zeros((model.cfg.n_layers, model.cfg.n_heads), device=model.cfg.device)
    count = 0
    for example in dataset:
        det_tokens = torch.tensor(
            [example.detection_sequence], dtype=torch.long, device=model.cfg.device
        )
        _, cache = model.run_with_cache(det_tokens)
        pair_index = torch.tensor(align_pairs, dtype=torch.long, device=model.cfg.device)
        q_index = pair_index[:, 0]
        k_index = pair_index[:, 1]
        for layer in range(model.cfg.n_layers):
            pattern = cache["pattern", layer][0]
            scores[layer] += pattern[:, q_index, k_index].mean(dim=-1)
        count += 1
    return scores / count


current_align = [(seq_len + i, i) for i in range(seq_len)]
corrected_align = [(seq_len + i, i + 1) for i in range(seq_len - 1)]

print("current_align  :", current_align)
print("corrected_align:", corrected_align)

current_scores = score_heads_with_alignment(model, dataset, current_align)
corrected_scores = score_heads_with_alignment(model, dataset, corrected_align)

current_align  : [(8, 0), (9, 1), (10, 2), (11, 3), (12, 4), (13, 5), (14, 6), (15, 7)]
corrected_align: [(8, 1), (9, 2), (10, 3), (11, 4), (12, 5), (13, 6), (14, 7)]


## 4. Side-by-side top-10 comparison

In [5]:
def ranking_frame(scores, label):
    rows = []
    for layer in range(scores.shape[0]):
        for head in range(scores.shape[1]):
            rows.append({
                "head_id": f"L{layer}H{head}",
                "layer": layer,
                "head": head,
                f"{label}_score": float(scores[layer, head]),
            })
    return (
        pd.DataFrame(rows)
        .sort_values(f"{label}_score", ascending=False)
        .head(10)
        .reset_index(drop=True)
    )


current_top10 = ranking_frame(current_scores, "current")
corrected_top10 = ranking_frame(corrected_scores, "corrected")

side_by_side = pd.concat(
    [
        current_top10[["head_id", "current_score"]].rename(columns={"head_id": "current_head"}),
        corrected_top10[["head_id", "corrected_score"]].rename(columns={"head_id": "corrected_head"}),
    ],
    axis=1,
)
side_by_side.index = [f"#{i + 1}" for i in range(len(side_by_side))]
side_by_side

,current_head,current_score,corrected_head,corrected_score
#1,L1H11,0.831878,L8H5,0.912908
#2,L3H3,0.668487,L6H2,0.826871
#3,L8H6,0.650688,L14H12,0.738365
#4,L3H5,0.646172,L6H9,0.724811
#5,L1H9,0.565435,L14H11,0.648437
#6,L2H7,0.547103,L8H1,0.629790
#7,L1H3,0.440447,L6H14,0.564265
#8,L1H1,0.392426,L9H3,0.531135
#9,L7H6,0.336015,L8H15,0.528056
#10,L4H13,0.273474,L9H4,0.474653


In [6]:
current_set = set(current_top10["head_id"])
corrected_set = set(corrected_top10["head_id"])

current_best = current_top10.iloc[0]["head_id"]
corrected_best = corrected_top10.iloc[0]["head_id"]

print(f"top-1 current   : {current_best}")
print(f"top-1 corrected : {corrected_best}")
print(f"top-1 changed?  : {current_best != corrected_best}")
print()
print(f"overlap of top-10 sets: {len(current_set & corrected_set)} / 10")
print(f"only in CURRENT top-10  : {sorted(current_set - corrected_set)}")
print(f"only in CORRECTED top-10: {sorted(corrected_set - current_set)}")
print()


def full_ranking(scores):
    rows = []
    for layer in range(scores.shape[0]):
        for head in range(scores.shape[1]):
            rows.append({"head_id": f"L{layer}H{head}", "score": float(scores[layer, head])})
    return pd.DataFrame(rows).sort_values("score", ascending=False).reset_index(drop=True)


# Where does the current top-1 detection head land under the corrected metric (and vice versa)?
corrected_full = full_ranking(corrected_scores)
current_full = full_ranking(current_scores)

cur_in_corr = corrected_full.index[corrected_full["head_id"] == current_best][0] + 1
corr_in_cur = current_full.index[current_full["head_id"] == corrected_best][0] + 1
cur_in_corr_score = corrected_full.loc[corrected_full["head_id"] == current_best, "score"].iloc[0]
corr_in_cur_score = current_full.loc[current_full["head_id"] == corrected_best, "score"].iloc[0]

print(f"{current_best} (top under CURRENT) ranks #{cur_in_corr} under CORRECTED "
      f"(score={cur_in_corr_score:.4f})")
print(f"{corrected_best} (top under CORRECTED) ranks #{corr_in_cur} under CURRENT "
      f"(score={corr_in_cur_score:.4f})")

top-1 current   : L1H11
top-1 corrected : L8H5
top-1 changed?  : True

overlap of top-10 sets: 0 / 10
only in CURRENT top-10  : ['L1H1', 'L1H11', 'L1H3', 'L1H9', 'L2H7', 'L3H3', 'L3H5', 'L4H13', 'L7H6', 'L8H6']
only in CORRECTED top-10: ['L14H11', 'L14H12', 'L6H14', 'L6H2', 'L6H9', 'L8H1', 'L8H15', 'L8H5', 'L9H3', 'L9H4']

L1H11 (top under CURRENT) ranks #376 under CORRECTED (score=0.0024)
L8H5 (top under CORRECTED) ranks #367 under CURRENT (score=0.0351)


## 5. Interpretation

_Read this together with the printed comparison above._

**What to look for.** The two metrics differ only in the key column of `align_pairs`:

- The **current** metric scores attention from the second copy of a token back to *itself* — the
  definition of a **duplicate-token head**.
- The **corrected** metric scores attention from the second copy of a token back to the token that
  *followed* it — the definition of a true **induction head** (Olsson et al. 2022).

**If the top heads shift** (e.g. `L1H11` drops out of the top of the corrected ranking and a
different head rises to #1): this is evidence that the current detection metric has been surfacing a
duplicate-token head rather than an induction head. The benchmark's headline "top detection head"
would then be mislabeled, and the heads near the top of the *corrected* ranking are the better
candidates for the induction circuit. Crucially, this would also explain a finding already baked
into `benchmark.py`: that the top *detection* head is not the best *causal* head. A duplicate-token
head can score high on the current attention metric while contributing little to the repetition
benefit on the held-out target — exactly the dissociation the causal search was built to work
around. The corrected detection metric may bring detection and causal rankings back into alignment.

**If the top heads do *not* shift** (the same heads, in roughly the same order, top both rankings):
then on this small synthetic family the duplicate-token and induction signals are carried by the
same heads, and the off-by-one does not change which heads the pipeline selects. The metric would
still be technically mislabeled relative to the Olsson definition, but it would not be the cause of
the detection/causal mismatch, and the practical conclusions of the existing run would stand.

**Caveats.** This is one configuration (`n_prompts=6, seq_len=8, seed=0`) on the synthetic family
only. The corrected alignment also has one fewer pair (`seq_len - 1` vs `seq_len`) because the last
token of the first copy has no successor to point at; that is expected and does not bias the mean.
Any change adopted here should be re-checked across seeds and on the `code` family before editing
`benchmark.py`.

## 6. Head-to-head causal comparison: L1H11 vs L8H5 vs L8H6 vs L14H11

The corrected detection metric reshuffles the *detection* ranking, and because `benchmark.py` draws its causal candidate pool from the detection ranking, some heads are never causally evaluated under the new pipeline. In particular **L8H6** — the previously reported top causal head — fell outside the corrected detection top-k and was never re-evaluated, while the new pipeline now reports **L14H11**.

A head's zero-ablation causal effect does **not** depend on the detection metric, so we can settle this directly: evaluate all four heads of interest under the *same* clean baseline and compare their `causal_selective_drop` head-to-head.

- **L1H11** — top *duplicate-token* detection head; included only to confirm low causal relevance.
- **L8H5** — top *induction* detection head under the corrected metric.
- **L8H6** — previously reported top causal head (now outside the detection candidate pool).
- **L14H11** — head the corrected pipeline now auto-selects as top causal.

In [7]:
from scripts.benchmark import mean_target_logprob

# Reuse the model + dataset already built above (synthetic, n_prompts=6, seq_len=8, seed=0).
# Rebuild defensively in case this cell is run in isolation.
try:
    model
    dataset
except NameError:
    model = load_model(device="cpu")
    dataset = build_synthetic_prompt_dataset(model, n_prompts=6, seq_len=8, seed=0)

# Clean (no-ablation) baselines — computed once, shared across all heads.
repeated_clean = mean_target_logprob(model, dataset, "repeated")
control_clean = mean_target_logprob(model, dataset, "control")
print(f"clean repeated target logprob : {repeated_clean:.4f}")
print(f"clean control  target logprob : {control_clean:.4f}")
print(f"behavioral gain (repeated - control): {repeated_clean - control_clean:.4f}\n")

heads_to_eval = {
    "L1H11": (1, 11),   # top duplicate-token head (not part of induction narrative)
    "L8H5": (8, 5),     # top induction detection head (corrected metric)
    "L8H6": (8, 6),     # previously reported top causal head
    "L14H11": (14, 11), # head auto-selected by the corrected pipeline
}

causal_rows = []
for head_id, (layer, head) in heads_to_eval.items():
    repeated_ablated = mean_target_logprob(model, dataset, "repeated", ablate_head=(layer, head))
    control_ablated = mean_target_logprob(model, dataset, "control", ablate_head=(layer, head))
    repeated_drop = repeated_clean - repeated_ablated
    control_drop = control_clean - control_ablated
    causal_rows.append({
        "head": head_id,
        "repeated_ablation_drop": repeated_drop,
        "control_ablation_drop": control_drop,
        "causal_selective_drop": repeated_drop - control_drop,
    })

causal_table = (
    pd.DataFrame(causal_rows)
    .sort_values("causal_selective_drop", ascending=False)
    .reset_index(drop=True)
)
causal_table

clean repeated target logprob : -2.4416
clean control  target logprob : -9.9971
behavioral gain (repeated - control): 7.5554



,head,repeated_ablation_drop,control_ablation_drop,causal_selective_drop
0,L14H11,0.032406,-0.050368,0.082774
1,L8H6,0.068774,-0.012211,0.080986
2,L8H5,-0.003891,-0.000931,-0.002960
3,L1H11,-0.012033,0.068349,-0.080382


### Interpretation

Measured on the baseline synthetic run (`n_prompts=6, seq_len=8, seed=0`, CPU); ablation is single-head zero-ablation, so these numbers do **not** depend on the detection metric:

| head | repeated_ablation_drop | control_ablation_drop | causal_selective_drop |
|---|---:|---:|---:|
| **L14H11** | 0.0324 | -0.0504 | **0.0828** |
| **L8H6** | 0.0688 | -0.0122 | **0.0810** |
| L8H5 | -0.0039 | -0.0009 | -0.0030 |
| L1H11 | -0.0120 | 0.0683 | -0.0804 |

**Which head is strongest?** By the headline `causal_selective_drop`, **L14H11 (0.0828) and L8H6 (0.0810) are effectively tied** — a gap of ~0.002, well within what we'd expect to flip across seeds. The corrected pipeline reports L14H11 only because the detection re-ranking happened to admit L14H11 and exclude L8H6 from the candidate pool; the head-to-head shows the two are causally comparable, not that L14H11 is meaningfully better.

**The decomposition actually favors L8H6 as the cleaner induction contributor.** `causal_selective_drop` is `repeated_drop − control_drop`, and the two heads earn their score very differently:

- **L8H6** has the largest **repeated_ablation_drop (0.0688)** of any head here — ablating it directly damages the model's ability to exploit the repeated prefix, which is the *direct* signature of an induction/copy mechanism. Its control drop is near zero (-0.0122).
- **L14H11** has a much smaller repeated effect (0.0324); roughly *half* of its selectivity comes from its **control_ablation_drop being negative (-0.0504)** — i.e. ablating it slightly *helps* the control condition. That inflates the difference score without reflecting a stronger repeated-copy effect.

So if we care about the head that most directly drives the repeated-context benefit, **L8H6 remains the better single-head causal candidate**, and it is not displaced by the metric fix — it was simply hidden by the detection-driven candidate selection.

**What this means for the induction-head narrative:**

1. **Detection and causation still dissociate — even after the fix.** L8H5 is now the top *induction-detection* head (attention score 0.91), yet its causal selective drop is ~0 (-0.0030): a strong induction *attention pattern* with little measured effect on the target logit. The corrected metric fixed the *labeling* of detection (L1H11 was a duplicate-token head, not induction) but did **not** collapse detection and causation into the same head. This vindicates `benchmark.py`'s core design choice of scoring causal effect independently of detection.
2. **L1H11 is confirmed causally irrelevant to induction.** Despite topping the *duplicate-token* detection metric, its causal_selective_drop is **negative** (-0.0804): ablating it helps the control more than the repeated condition. It belongs to a duplicate-token story, not the induction-copy story, and is included here only as a negative control.
3. **The causal conclusion from before the fix stands.** L8H6 is still a genuine, leading single-head causal contributor to the repeated-context effect. The off-by-one correction changed *who gets detected*, not *who does the causal work*. (Multi-head ablations remain materially stronger than any single head, so the fuller causal story is still a small cooperating set rather than one head.)

**Caveats:** single config, single seed, single-head ablations only; the L14H11/L8H6 ordering is within noise and should be re-checked across seeds before treating either as definitively "the" top causal head.